# 개요

안녕하세요. 이 페이지는 제가 진행하면서 필요한 내용을 테스트하는 곳이에요.

In [12]:
# !pip install pymupdf

In [13]:
# !pip install pymupdf openai

In [ ]:
# !pip install chromadb openai pymupdf

In [ ]:
# !pip install pymupdf

# 1. 검색 쿼리 생성 프롬프트 + GPT 호출

In [14]:
import json
from openai import OpenAI

client = OpenAI()

SEARCH_QUERY_PROMPT = """
You are an expert search-query generator designed for RAG (Retrieval-Augmented Generation) systems.
Your objective is to generate the most effective query_text for retrieving Korean-language documents
from a vector database.

You will receive a JSON payload that includes:
- country_code (ISO country code)
- hs_code (10-digit HS code)
- system_generated metadata (date range)
- default_sections (mandatory analysis areas)
- user_requirements (boolean analysis options)

Your task:
1. Always generate the final query_text in Korean.
2. Start the query with: “{country name in Korean} 시장 HS {hs_code}”.
Use the exact value of country_name_kr provided inside the payload.
Do NOT translate or infer the country name from country_code.
3. Always include the mandatory analysis categories from default_sections using a compressed phrase:
   “시장규모·수출현황·트렌드·전망·PEST·SWOT·전략”
4. For every TRUE field in user_requirements, append the corresponding Korean label:
   - include_market_risk → “시장 리스크”
   - include_price_trend → “가격 추세”
   - include_competitor_analysis → “경쟁국 분석”
   - include_supply_chain_analysis → “공급망 분석”
   - include_regulation_review → “규제 검토”
   - include_demand_forecast → “수요전망”
   - include_policy_impact → “정책 영향”
   - include_scenario_risk → “시나리오 리스크”
   - include_trade_graphs → “교역 그래프”
   - include_monthly_trends → “월별 추세”
   - include_yearly_graphs → “연도별 그래프”
   - include_top_partners → “주요 교역국 비교”
   - include_price_index_graph → “가격지수”
   - include_volume_vs_value → “수량·금액 비교”
   - include_table_summary → “요약 표”
5. If custom_request_text is provided, append it at the end.
6. Combine all elements with “ / ” as separators.
7. Output only the final query_text. Do NOT include explanations.

Important safety rule:
Do NOT invent or assume any information that is not explicitly included in the payload.
Do NOT add details, facts, years, figures, examples, or assumptions beyond what the payload provides.
You must strictly use only the information explicitly given in the payload when generating the query_text.
If the payload lacks information, do NOT fill the gap—simply omit that part.
"""


# 2. PDF → 텍스트 추출 함수

In [15]:
import fitz

def extract_pdf_text_all(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    texts = []

    for page in doc:
        t = page.get_text("text")
        if t:
            texts.append(t)

    doc.close()
    return "\n".join(texts)


def extract_pdf_page_text(pdf_path: str, page_num: int) -> str:
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    text = page.get_text("text")
    doc.close()
    return text


# 3. 텍스트 → 문단 분리 함수

In [31]:
def split_into_paragraphs(text: str):
    raw_paragraphs = text.split("\n\n")
    paragraphs = []

    for p in raw_paragraphs:
        cleaned = p.strip()
        if cleaned:
            paragraphs.append(cleaned)
    return paragraphs

pdf_path = "./data/kang/2025 미국 진출전략.pdf"


# 4. 문단 → 청킹 함수

In [32]:
def chunk_paragraphs(paragraphs, chunk_size: int, chunk_overlap: int):
    chunks = []
    current = ""

    for p in paragraphs:
        if not current:
            candidate = p
        else:
            candidate = current + "\n\n" + p

        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)

            if len(p) > chunk_overlap:
                current = p[-chunk_overlap:]
            else:
                current = p

    if current:
        chunks.append(current)

    return chunks


# 5. 청킹 규칙

In [33]:
CHUNK_RULES = {
    "country_info": {
        "US":  {"chunk_size": 700,  "chunk_overlap": 150},
        "VN":  {"chunk_size": 1000, "chunk_overlap": 200},
        "JP":  {"chunk_size": 1000, "chunk_overlap": 200},
    },
    "strategy": {
        "2025": {
            "US": {"chunk_size": 1000, "chunk_overlap": 200},
            "VN": {"chunk_size": 1000, "chunk_overlap": 200},
            "JP": {"chunk_size": 1000, "chunk_overlap": 200},
        },
        "2024": {
            "US": {"chunk_size": 1000, "chunk_overlap": 180},
            "VN": {"chunk_size": 1000, "chunk_overlap": 180},
            "JP": {"chunk_size": 1000, "chunk_overlap": 200},
        },
        "2023": {
            "US": {"chunk_size": 1000, "chunk_overlap": 180},
            "VN": {"chunk_size": 1000, "chunk_overlap": 180},
            "JP": {"chunk_size": 1000, "chunk_overlap": 180},
        },
    }
}


def get_chunk_params(doc_type: str, country_code: str, year: int):
    if doc_type == "country_info":
        return CHUNK_RULES["country_info"].get(country_code)
    elif doc_type == "strategy":
        return CHUNK_RULES["strategy"].get(str(year), {}).get(country_code)
    return None


# 6. 메타데이터 생성 및 JSONL 저장

In [34]:
def save_chunks_to_jsonl(chunks, metadata, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        for i, chunk in enumerate(chunks):

            enriched_metadata = metadata.copy()
            enriched_metadata["chunk_index"] = i

            record = {
                "id": f"{metadata['country_code']}_{metadata['year']}_{i}",
                "text": chunk,
                "metadata": enriched_metadata
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("Saved:", save_path)


# 7. 국가명 매핑 테이블

In [35]:
COUNTRY_NAME_KR = {
    "US": "미국",
    "JP": "일본",
    "VN": "베트남",
    "CN": "중국",
    "DE": "독일",
    "FR": "프랑스",
    "GB": "영국",
    "IN": "인도",
    "CA": "캐나다",
}

def enrich_payload(payload):
    code = payload.get("country_code")
    payload["country_name_kr"] = COUNTRY_NAME_KR.get(code, code)
    return payload


# 8. RAG 전처리 & 검색 쿼리 생성

In [36]:
def build_rag_query(payload):
    # enrich payload with Korean country name
    payload = enrich_payload(payload)

    messages = [
        {"role": "system", "content": SEARCH_QUERY_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}
    ]

    response = client.chat.completions.create(
        model="gpt-5",
        temperature=0,
        messages=messages
    )

    query_text = response.choices[0].message["content"].strip()

    filters = {
        "country_code": payload["country_code"],
        "hs10": payload["hs_code"]
    }

    return filters, query_text


In [28]:
import os

folder_path = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang"

print(os.path.exists(folder_path))
os.listdir(folder_path)


True


['2025 미국 진출전략.pdf']

In [29]:
pdf_path = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang\2025 미국 진출전략.pdf"

import os
print(os.path.exists(pdf_path))


True


In [30]:
text = extract_pdf_text_all(pdf_path)

print("텍스트 길이:", len(text))
print(text[:1000])

텍스트 길이: 138262
Ⅰ. 시장 평가 및 주요이슈
3
1. 개요
가. 시장 전망
□확장세는 둔화되나 침체 없는 경제 성장 전망
⚫고강도 긴축정책에 따른 침체 우려에도 불구, 미 경제는 강력한 성장세를 이어왔으나 그간 누적된 통화 
긴축의 효과로 2024년 하반기부터 성장세 둔화
- 2025년 미국 경제성장률은 1%대 후반 혹은 2% 초반대를 기록할 것으로 전망
- 2024년 9월 연방준비제도(Fed·연준)가 금리 인하 사이클을 시작하면서 인플레이션 억제를 위해 그간 
이어온 통화 긴축 정책도 종료
   * 연준은 ’24년 9월과 11월 FOMC에서 각각 50bp와 25bp 기준금리를 인하해 ’24년 11월 현재 미국의 기준금리 구간은 
4.5∼4.75%   
- 이에 따라 미국이 현재의 경제 확장세를 유지하고, 목표했던 경제 연착륙 도달 가능성도 상승
- 디스인플레이션이 추세적으로 진행되고 있으며, 노동시장의 수급 불균형이 상당 부분 해소되고, 점진적 
냉각과 임금 상승세 둔화 전망
   * 미 농업 부문 신규 고용(만 명): (’23.12월) 29.0, (’24.5월) 21.6, (6월) 11.8, (7월) 14.4, (8월) 7.8, (9월) 22.3, (10월) 
1.2 (미 노동부, ’24.11.)
- 강경 이민 정책과 수입품에 대폭 관세 인상을 예고한 트럼프 전 대통령이 47대 대통령으로 당선되면서 
인플레이션 재발 우려 확산
   * 무디스는 트럼프 당선 이후 ’25년 미 인플레이션 전망치를 2.4%(9월)에서 최소 3%로 상향 조정 
Ⅰ
시장 평가 및 주요 이슈

4
<2024∼2026년 미국 경제 전망> 
(단위: %)
자료: 미 의회예산국(2024년 6월)
나. 주요 경제지표
주 요 지 표 
단   위
2018년
2019년
2020년
2021년
2022년
2023년
2024년
2025년
인구
백만 명
327
329
331
332
334
340
342
344
명목GDP
십억$ 
20,657
21,540
21,354
23,681
26,007


In [37]:
import os
import fitz
import json


# --------------------------------------------
# 1. PDF 텍스트 추출 함수
# --------------------------------------------
def extract_pdf_text_all(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    texts = []

    for page in doc:
        text = page.get_text("text")
        if text:
            texts.append(text)

    doc.close()
    return "\n".join(texts)


# --------------------------------------------
# 2. 문단 분리 함수
# --------------------------------------------
def split_into_paragraphs(text: str):
    raw_paragraphs = text.split("\n\n")  
    paragraphs = [p.strip() for p in raw_paragraphs if p.strip()]
    return paragraphs


# --------------------------------------------
# 3. 청킹 규칙
# --------------------------------------------
CHUNK_RULES = {
    "country_info": {
        "US":  {"chunk_size": 700,  "chunk_overlap": 150},
        "VN":  {"chunk_size": 1000, "chunk_overlap": 200},
        "JP":  {"chunk_size": 1000, "chunk_overlap": 200},
    },
    "strategy": {
        "2025": {
            "US": {"chunk_size": 1000, "chunk_overlap": 200},
            "VN": {"chunk_size": 1000, "chunk_overlap": 200},
            "JP": {"chunk_size": 1000, "chunk_overlap": 200},
        },
        "2024": {
            "US": {"chunk_size": 1000, "chunk_overlap": 180},
            "VN": {"chunk_size": 1000, "chunk_overlap": 180},
            "JP": {"chunk_size": 1000, "chunk_overlap": 200},
        },
        "2023": {
            "US": {"chunk_size": 1000, "chunk_overlap": 180},
            "VN": {"chunk_size": 1000, "chunk_overlap": 180},
            "JP": {"chunk_size": 1000, "chunk_overlap": 180},
        },
    }
}


def get_chunk_params(doc_type: str, country_code: str, year: int):
    year_str = str(year)
    if doc_type == "country_info":
        return CHUNK_RULES["country_info"][country_code]
    else:
        return CHUNK_RULES["strategy"][year_str][country_code]


# --------------------------------------------
# 4. 청킹 함수
# --------------------------------------------
def chunk_paragraphs(paragraphs, chunk_size, chunk_overlap):
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) + 1 <= chunk_size:
            current += p + " "
        else:
            chunks.append(current.strip())
            current = p + " "

    if current.strip():
        chunks.append(current.strip())

    return chunks


# --------------------------------------------
# 5. JSONL 저장 함수
# --------------------------------------------
def save_chunks_to_jsonl(chunks, metadata, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        for i, chunk in enumerate(chunks):
            item = {
                "id": f"{metadata['country_code']}_{metadata['year']}_{metadata['doc_type']}_{i}",
                "text": chunk,
                "metadata": {
                    "country_code": metadata["country_code"],
                    "year": metadata["year"],
                    "doc_type": metadata["doc_type"],
                    "source": metadata["source"],
                    "filename": metadata["filename"],
                    "chunk_index": i
                }
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"JSONL saved: {output_path}")


# --------------------------------------------
# 6. 전체 자동 실행 함수
# --------------------------------------------
def process_pdf_to_jsonl(pdf_path, country_code, year, doc_type):
    print("1) PDF 텍스트 추출 중...")
    text = extract_pdf_text_all(pdf_path)
    print("   텍스트 길이:", len(text))

    print("2) 문단 분리 중...")
    paragraphs = split_into_paragraphs(text)
    print("   문단 개수:", len(paragraphs))

    print("3) 청킹 파라미터 로딩...")
    params = get_chunk_params(doc_type, country_code, year)
    print("   사용 chunk_size:", params["chunk_size"])
    print("   사용 chunk_overlap:", params["chunk_overlap"])

    print("4) 청킹 실행...")
    chunks = chunk_paragraphs(paragraphs, params["chunk_size"], params["chunk_overlap"])
    print("   청크 개수:", len(chunks))

    print("5) JSONL 저장 중...")
    metadata = {
        "country_code": country_code,
        "year": year,
        "doc_type": doc_type,
        "source": "KOTRA",
        "filename": os.path.basename(pdf_path)
    }

    out_path = f"./processed_jsonl/{country_code}_{year}_{doc_type}.jsonl"
    save_chunks_to_jsonl(chunks, metadata, out_path)

    print("   완료!")
    return chunks


# --------------------------------------------
# 7. 실제 실행 예시
# --------------------------------------------
pdf_path = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang\2025 미국 진출전략.pdf"

chunks = process_pdf_to_jsonl(
    pdf_path=pdf_path,
    country_code="US",  
    year=2025,
    doc_type="strategy"
)


1) PDF 텍스트 추출 중...
   텍스트 길이: 138262
2) 문단 분리 중...
   문단 개수: 121
3) 청킹 파라미터 로딩...
   사용 chunk_size: 1000
   사용 chunk_overlap: 200
4) 청킹 실행...
   청크 개수: 120
5) JSONL 저장 중...
JSONL saved: ./processed_jsonl/US_2025_strategy.jsonl
   완료!


# 9.데이터 소스 청킹 및 임베딩 

In [ ]:
# KATI 제외 파일 청킹 및 저장 
import os
import fitz
import json

# 입력 폴더 (PDF 저장 위치)
INPUT_DIR = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang"

# 출력 폴더
OUTPUT_DIR = "./processed_jsonl_non_kati"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def extract_text_no_preprocess(pdf_path):
    """
    KATI 제외 PDF는 목차·표지 정리가 이미 됨.
    따라서 전처리 없이 페이지 전체 텍스트를 그대로 사용.
    """
    doc = fitz.open(pdf_path)
    total_text = ""

    for page in doc:
        total_text += page.get_text("text") + "\n"

    doc.close()
    return total_text


def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    chunks = []
    start = 0
    length = len(text)

    while start < length:
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)

    return chunks


def parse_filename_non_kati(filename):
    """
    파일명 종류 예:
    - 2025_Market_entry_strategy_US.pdf
    - national_Information_US.pdf

    output:
    { year, doc_type, country }
    """
    name = filename.replace(".pdf", "")
    parts = name.split("_")

    # 1) 국가정보 PDF
    if name.startswith("national_Information"):
        return {
            "year": None,
            "doc_type": "national_information",
            "country": parts[-1]
        }

    # 2) 진출전략 등 일반 PDF
    year = int(parts[0])
    doc_type = "_".join(parts[1:-1]).lower()
    country = parts[-1]

    return {
        "year": year,
        "doc_type": doc_type,
        "country": country
    }


def process_non_kati_pdfs():
    """
    KATI 제외 모든 PDF를:
    - 전처리 없이 텍스트 추출
    - 청킹
    - JSONL 저장
    """
    for file in os.listdir(INPUT_DIR):
        if not file.endswith(".pdf"):
            continue
        if "kati_info" in file:        # KATI 제외
            continue

        pdf_path = os.path.join(INPUT_DIR, file)
        print(f"Processing: {file}")

        meta = parse_filename_non_kati(file)

        # 1. 텍스트 추출
        text = extract_text_no_preprocess(pdf_path)

        # 2. 청킹
        chunks = chunk_text(text, chunk_size=1000, chunk_overlap=200)

        # 3. JSONL 경로 생성
        out_path = os.path.join(
            OUTPUT_DIR,
            f"{meta['doc_type']}_{meta['country']}_{meta['year']}.jsonl"
        )

        # 4. 저장
        with open(out_path, "w", encoding="utf-8") as f:
            for i, ch in enumerate(chunks):
                record = {
                    "id": f"{meta['doc_type']}_{meta['country']}_{meta['year']}_{i}",
                    "text": ch,
                    "metadata": {
                        "country": meta["country"],
                        "doc_type": meta["doc_type"],
                        "year": meta["year"],
                        "source": "PDF"
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f" → Saved {len(chunks)} chunks → {out_path}")


# 실행
process_non_kati_pdfs()


Processing: 2023_Market_entry_strategy_JP.pdf
 → Saved 123 chunks → ./processed_jsonl_non_kati\market_entry_strategy_JP_2023.jsonl
Processing: 2023_Market_entry_strategy_US.pdf
 → Saved 124 chunks → ./processed_jsonl_non_kati\market_entry_strategy_US_2023.jsonl
Processing: 2023_Market_entry_strategy_VN.pdf
 → Saved 197 chunks → ./processed_jsonl_non_kati\market_entry_strategy_VN_2023.jsonl
Processing: 2024_Market_entry_strategy_JP.pdf
 → Saved 123 chunks → ./processed_jsonl_non_kati\market_entry_strategy_JP_2024.jsonl
Processing: 2024_Market_entry_strategy_US.pdf
MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

 → Saved 181 chunks → ./processed_jsonl_non_kati\market_entry_strategy_US_2024.jsonl
Processing: 2024_Market_entry_strategy_VN.pdf
 → Saved 134 chunks → ./processed_jsonl

# 내가 가진 pdf에 토큰 수와 페이지 수를 파악해보자ㅓ

In [42]:
import os
import fitz
import tiktoken

folder = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang"

def get_pdf_page_count(pdf_path):
    doc = fitz.open(pdf_path)
    page_count = doc.page_count
    doc.close()
    return page_count

def get_pdf_token_count(pdf_path, model="gpt-4o-mini"):
    doc = fitz.open(pdf_path)
    
    full_text = ""
    for page in doc:
        full_text += page.get_text("text") + "\n"

    doc.close()

    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(full_text))


total_pages = 0
total_tokens = 0

for file in os.listdir(folder):
    if file.lower().endswith(".pdf"):
        path = os.path.join(folder, file)

        pages = get_pdf_page_count(path)
        tokens = get_pdf_token_count(path)

        total_pages += pages
        total_tokens += tokens

        print(f"{file} → {pages} pages / {tokens} tokens")

print("\n총 페이지 수:", total_pages)
print("총 토큰 수:", total_tokens)


2022 미국.pdf → 121 pages / 42676 tokens
2022년 베트남.pdf → 6 pages / 3397 tokens
2022년 일본.pdf → 145 pages / 99240 tokens
2023 미국 진출전략.pdf → 90 pages / 60052 tokens
MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

2023 미국.pdf → 64 pages / 43637 tokens
2023 베트남 진출전략.pdf → 146 pages / 97994 tokens
2023 베트남.pdf → 52 pages / 33329 tokens
2023 일본 진출전략.pdf → 91 pages / 61689 tokens
2023 일본.pdf → 55 pages / 36019 tokens
MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

MuPDF error: syntax error: cid font is missing descendant fonts

2024 미국 진출전략.pdf → 126 pages / 87404 tokens
2024 미국.pdf → 66 pages / 46516 tokens
2024 베트남 진출전략.pdf → 98 pages / 67695 tokens
2024 베트남.pdf → 57 pages / 36498 tokens
2024 일본 진출전략.pdf → 96 pages / 63159 tokens
2024 일본.pdf → 58 pages / 39009 tokens
2024 중국.pdf → 65 pages / 45436 toke

In [ ]:
# KATI만 전처리 후 청킹 및 저장 
import os
import fitz
import json

# 입력 폴더
INPUT_DIR = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang"

# 출력 폴더
OUTPUT_DIR = "./processed_jsonl_kati"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 제거 조건 키워드
REMOVE_KEYWORDS = [
    "목차", "Contents", "CONTENTS", "Table of Contents",
    "저작권", "Copyright",
    "Disclaimer", "참고문헌", "References",
]


def is_noise_page(text):
    """
    페이지가 노이즈(제거 대상)인지 판단하는 함수.
    기준:
    1) 텍스트 길이가 너무 짧음 (표지/공백/이미지 등)
    2) 목차/저작권/참고문헌 등 키워드 포함
    """
    if len(text.strip()) < 80:
        return True

    for kw in REMOVE_KEYWORDS:
        if kw.lower() in text.lower():
            return True

    return False


def extract_clean_text(pdf_path):
    """
    KATI PDF에서 노이즈 페이지를 제거하고
    본문 텍스트만 추출하는 함수.
    """
    doc = fitz.open(pdf_path)
    cleaned_text = ""

    for page in doc:
        text = page.get_text("text")
        if is_noise_page(text):
            continue
        cleaned_text += text + "\n"

    doc.close()
    return cleaned_text


def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    """텍스트를 chunk_size 기준으로 잘라 리스트로 반환."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)
    return chunks


def parse_kati_filename(filename):
    """
    파일명 예: 2024_kati_info_US.pdf
    → year: 2024
    → doc_type: kati_info
    → country: US
    """
    name = filename.replace(".pdf", "")
    parts = name.split("_")

    year = int(parts[0])
    doc_type = parts[1] + "_" + parts[2]     # kati_info
    country = parts[3]

    return year, doc_type, country


def process_kati_pdfs():
    """
    data/kang 폴더 안 모든 KATI PDF 처리 자동화:
    - 전처리
    - 청킹
    - 메타데이터 추가
    - JSONL 저장
    """
    for file in os.listdir(INPUT_DIR):
        if not file.endswith(".pdf"):
            continue
        if "kati_info" not in file:
            continue

        pdf_path = os.path.join(INPUT_DIR, file)
        print(f"Processing: {file}")

        # 파일명 파싱
        year, doc_type, country = parse_kati_filename(file)

        # 1) 전처리 → 노이즈 페이지 제거
        clean_text = extract_clean_text(pdf_path)

        # 2) 청킹
        chunks = chunk_text(clean_text, chunk_size=1000, chunk_overlap=200)

        # 3) JSONL 파일명
        out_path = os.path.join(
            OUTPUT_DIR,
            f"{year}_{country}_{doc_type}.jsonl"
        )

        # 4) JSONL 저장
        with open(out_path, "w", encoding="utf-8") as f:
            for i, ch in enumerate(chunks):
                record = {
                    "id": f"{year}_{country}_{doc_type}_{i}",
                    "text": ch,
                    "metadata": {
                        "year": year,
                        "country": country,
                        "doc_type": doc_type,
                        "source": "KATI_PDF"
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f" → Saved {len(chunks)} chunks to {out_path}")


# 실행
process_kati_pdfs()


Processing: 2022_kati_info_JP.pdf
 → Saved 187 chunks to ./processed_jsonl_kati\2022_JP_kati_info.jsonl
Processing: 2022_kati_info_US.pdf
 → Saved 85 chunks to ./processed_jsonl_kati\2022_US_kati_info.jsonl
Processing: 2022_kati_info_VN.pdf
 → Saved 7 chunks to ./processed_jsonl_kati\2022_VN_kati_info.jsonl
Processing: 2023_kati_info_JP.pdf
 → Saved 68 chunks to ./processed_jsonl_kati\2023_JP_kati_info.jsonl
Processing: 2023_kati_info_US.pdf
MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

 → Saved 86 chunks to ./processed_jsonl_kati\2023_US_kati_info.jsonl
Processing: 2023_kati_info_VN.pdf
 → Saved 63 chunks to ./processed_jsonl_kati\2023_VN_kati_info.jsonl
Processing: 2024_kati_info_JP.pdf
 → Saved 74 chunks to ./processed_jsonl_kati\2024_JP_kati_info.jsonl
Processing: 2024_kati_info_US.pdf
 → Saved 93 chunks to ./processed_jsonl_kati\2024_US_kati_info.jsonl
Processing: 2024_kati_info_VN.pdf
 → Saved 71 chunks to ./processed_jsonl_kati\2